# 01 — Verify Pipeline

Smoke test: load catalog data, configure BacktestEngine, run NT's built-in EMACross, verify fills.
If the final cell shows fills, the pipeline works end-to-end.

In [ ]:
# Cell 1: Imports + catalog load
from nautilus_trader.persistence.catalog import ParquetDataCatalog

from src.core import bar_type_str

INSTRUMENT_ID = "BTC-USD-PERP.HYPERLIQUID"
BAR_TYPE_STR  = bar_type_str(INSTRUMENT_ID, "1h")

catalog = ParquetDataCatalog("../data/catalog")
print("Data types:", catalog.list_data_types())
print("Instruments:", [str(i.id) for i in catalog.instruments()])

In [ ]:
# Cell 2: Load BTC 1h bars + verify ts_init_delta
import pandas as pd

bars = catalog.bars(bar_types=[BAR_TYPE_STR])
print(f"Bar count: {len(bars)}")

first = bars[0]
last = bars[-1]
print(f"First: {pd.Timestamp(first.ts_event, unit='ns', tz='UTC')}")
print(f"Last:  {pd.Timestamp(last.ts_event, unit='ns', tz='UTC')}")

# Verify ts_init_delta (critical for avoiding look-ahead bias)
print(f"\nFirst bar ts_event: {first.ts_event}")
print(f"First bar ts_init:  {first.ts_init}")
delta_ns = first.ts_init - first.ts_event
print(f"Delta (ns): {delta_ns}")
assert delta_ns == 3_600_000_000_000, f"ts_init_delta wrong! Expected 3.6T ns, got {delta_ns}"
print("ts_init_delta OK (1 hour)")

In [ ]:
# Cell 3: Configure engine
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.backtest.config import BacktestEngineConfig
from nautilus_trader.config import LoggingConfig
from nautilus_trader.model.enums import AccountType, OmsType
from nautilus_trader.model.identifiers import Venue
from nautilus_trader.model.objects import Money
from nautilus_trader.model.currencies import USDC

engine = BacktestEngine(config=BacktestEngineConfig(
    logging=LoggingConfig(log_level="ERROR"),
))

engine.add_venue(
    venue=Venue("HYPERLIQUID"),
    oms_type=OmsType.NETTING,
    account_type=AccountType.MARGIN,
    base_currency=None,
    starting_balances=[Money(100_000, USDC)],
)

instrument = catalog.instruments(instrument_ids=[INSTRUMENT_ID])[0]
engine.add_instrument(instrument)
engine.add_data(bars)

print(f"Engine configured. Instrument: {instrument.id}")

In [ ]:
# Cell 4: Run with NT's built-in EMACross
from decimal import Decimal
from nautilus_trader.examples.strategies.ema_cross import EMACross, EMACrossConfig
from nautilus_trader.model.data import BarType

config = EMACrossConfig(
    instrument_id=instrument.id,
    bar_type=BarType.from_str(BAR_TYPE_STR),
    trade_size=Decimal("0.01"),
    fast_ema_period=10,
    slow_ema_period=20,
)
strategy = EMACross(config=config)
engine.add_strategy(strategy)

engine.run()
print("Engine run complete.")

In [ ]:
# Cell 5: Print fills — if rows exist, pipeline is verified
fills = engine.trader.generate_order_fills_report()
print(f"Total fills: {len(fills)}")

if len(fills) > 0:
    print("\nPIPELINE VERIFIED — fills generated.")
    display(fills.head(10))
else:
    print("\nWARNING: No fills generated. Check strategy and data.")

engine.dispose()